In [1]:
import requests

url = "https://api.frankfurter.dev/v2/providers/ecb/rates"
r = requests.get(url, params={"date": "2024-05-15", "quotes": "usd,gbp"}, timeout=10)
r.status_code, r.json()

(200,
 [{'date': '2024-05-15', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.8584},
  {'date': '2024-05-15', 'base': 'EUR', 'quote': 'USD', 'rate': 1.0832}])

In [2]:
r = requests.get(
    url,
    params={"from": "2024-01-01", "to": "2024-01-31", "quotes": "usd"},
    timeout=10,
)
data = r.json()
print(r.status_code, len(data))
data[:3]

200 23


[{'date': '2023-12-29', 'base': 'EUR', 'quote': 'USD', 'rate': 1.105},
 {'date': '2024-01-02', 'base': 'EUR', 'quote': 'USD', 'rate': 1.0956},
 {'date': '2024-01-03', 'base': 'EUR', 'quote': 'USD', 'rate': 1.0919}]

In [3]:
r = requests.get(url, params={"date": "2024-05-18", "quotes": "usd"}, timeout=10)
r.status_code, r.json()

(200, [{'date': '2024-05-17', 'base': 'EUR', 'quote': 'USD', 'rate': 1.0844}])

In [4]:
r = requests.get(url, params={"date": "2024-05-15", "quotes": "xxx"}, timeout=10)
r.status_code, r.text

(422, '{"status":422,"message":"invalid currency: XXX"}')

In [5]:
from datetime import date

from trade_pipeline.clients.fx import fetch_ecb_rates, parse_rates
from trade_pipeline.logging_setup import setup_logging

setup_logging("INFO")



In [6]:
raw = fetch_ecb_rates(date(2024, 1, 1), date(2024, 1, 31), ["usd"])
rates = parse_rates(raw, date(2024, 1, 1), date(2024, 1, 31))
len(raw), len(rates), rates[0]

2026-11-15 13:07:24,466 | INFO | trade_pipeline.clients.fx | Requesting ECB rates 2024-01-01..2024-01-31 for ['usd']
2026-11-15 13:07:24,766 | INFO | trade_pipeline.clients.fx | Received 23 rows


(23,
 22,
 {'date': datetime.date(2024, 1, 2),
  'base': 'EUR',
  'quote': 'USD',
  'rate': 1.0956})

In [7]:
import requests

from trade_pipeline.config import load_settings

settings = load_settings()

COMTRADE_URL = "https://comtradeapi.un.org/data/v1/get/C/A/HS"
headers = {"Ocp-Apim-Subscription-Key": settings.comtrade_api_key}

params = {
    "reporterCode": "276",   # Германия сообщает
    "partnerCode": "156",    # о торговле с Китаем
    "flowCode": "X",         # экспорт
    "cmdCode": "27",         # HS 27, минеральное топливо
    "period": "2023",
}

r = requests.get(COMTRADE_URL, headers=headers, params=params, timeout=30)
data = r.json()
print(r.status_code, data.get("count"), data.get("error"))
data["data"][:2]

200 24 


[{'typeCode': 'C',
  'freqCode': 'A',
  'refPeriodId': 20230101,
  'refYear': 2023,
  'refMonth': 52,
  'period': '2023',
  'reporterCode': 276,
  'reporterISO': None,
  'reporterDesc': None,
  'flowCode': 'X',
  'flowDesc': None,
  'partnerCode': 156,
  'partnerISO': None,
  'partnerDesc': None,
  'partner2Code': 0,
  'partner2ISO': None,
  'partner2Desc': None,
  'classificationCode': 'H6',
  'classificationSearchCode': 'HS',
  'isOriginalClassification': True,
  'cmdCode': '27',
  'cmdDesc': None,
  'aggrLevel': None,
  'isLeaf': None,
  'customsCode': 'C03',
  'customsDesc': None,
  'mosCode': '0',
  'motCode': 9200,
  'motDesc': None,
  'qtyUnitCode': -1,
  'qtyUnitAbbr': None,
  'qty': 0.0,
  'isQtyEstimated': False,
  'altQtyUnitCode': -1,
  'altQtyUnitAbbr': None,
  'altQty': 0.0,
  'isAltQtyEstimated': False,
  'netWgt': 0.0,
  'isNetWgtEstimated': True,
  'grossWgt': 0.0,
  'isGrossWgtEstimated': False,
  'cifvalue': None,
  'fobvalue': 1839.095,
  'primaryValue': 1839.095,
 

In [8]:
print("status:", r.status_code, "rows:", data["count"])
for row in data["data"]:
    print(
        row["customsCode"],
        row["motCode"],
        row["partner2Code"],
        row["isAggregate"],
        row["primaryValue"],
    )

status: 200 rows: 24
C03 9200 0 True 1839.095
C04 1000 0 True 2339.382
C03 3100 0 True 7246246.647
C03 9200 899 True 1839.095
C00 9200 0 True 1839.095
C00 1000 0 True 6113129.926
C00 3100 899 True 7246246.647
C00 2100 899 True 162897148.963
C04 0 899 True 2339.382
C03 1000 0 True 6110790.544
C00 0 0 True 176258364.631
C03 0 899 True 176256025.249
C04 0 0 True 2339.382
C03 3100 899 True 7246246.647
C03 2100 899 True 162897148.963
C00 3100 0 True 7246246.647
C03 2100 0 True 162897148.963
C00 9200 899 True 1839.095
C00 0 899 True 176258364.631
C03 0 0 True 176256025.249
C00 2100 0 True 162897148.963
C00 1000 899 True 6113129.926
C04 1000 899 True 2339.382
C03 1000 899 True 6110790.544


In [9]:
total = sum(row["primaryValue"] for row in data["data"])
true_total = next(
    row["primaryValue"] for row in data["data"]
    if row["customsCode"] == "C00" and row["motCode"] == 0 and row["partner2Code"] == 0
)
total, true_total, total / true_total

(1410066917.048, 176258364.631, 8.0)

Ловушка: зерно данных. На один запрос API возвращает 24 строки: одна и та же торговля нарезана по таможенному режиму, виду транспорта и второму партнёру во всех комбинациях. Сумма всех строк завышает торговлю в 8 раз (1,41 млрд вместо 176 млн $). Правильный итог — строка customsCode=C00, motCode=0, partner2Code=0. Решение: запрашивать только итог и проверять, что на ключ пришла ровно одна строка.

In [10]:
from pathlib import Path

from trade_pipeline.clients.comtrade import fetch_trade, parse_trade, save_raw

payload = fetch_trade(settings.comtrade_api_key, "276", "156", "X", "27", "2023")
save_raw(payload, Path("../data/raw/comtrade/276_156_X_27_2023.json"))
parse_trade(payload)

2026-11-15 13:07:31,602 | INFO | trade_pipeline.clients.comtrade | Comtrade {'reporterCode': '276', 'partnerCode': '156', 'flowCode': 'X', 'cmdCode': '27', 'period': '2023'}: 24 rows
2026-11-15 13:07:31,610 | INFO | trade_pipeline.clients.comtrade | Saved raw payload to ../data/raw/comtrade/276_156_X_27_2023.json


[{'reporter_code': 276,
  'partner_code': 156,
  'flow_code': 'X',
  'cmd_code': '27',
  'period': 2023,
  'primary_value': 176258364.631,
  'fob_value': 176258364.631,
  'cif_value': None}]

Comtrade иногда не отвечает за 30 с → в клиенте нужны retries с backoff

Первый mirror gap. HS 27, 2023: экспорт DE→CN 176,26 млн $ (FOB), импорт CN←DE 183,44 млн $ (CIF), разрыв +4,1%, в пределах нормальной надбавки CIF/FOB. Страны отчитываются с разной детализацией (DE — 24 строки, CN — 1), поэтому берём итоговую строку и проверяем, что она единственная.

In [11]:
from trade_pipeline.db import upsert_trade_flows

rows = parse_trade(payload)
upsert_trade_flows(settings.database_url, rows)
upsert_trade_flows(settings.database_url, rows)

2026-11-15 13:07:31,709 | INFO | trade_pipeline.db | Upserted 1 trade rows
2026-11-15 13:07:31,719 | INFO | trade_pipeline.db | Upserted 1 trade rows


1

Добавь раздел «Mirror gaps: первые результаты» с этой таблицей и тремя выводами: контроль в норме, крупные разрывы сосредоточены в HS 71, следующий шаг — детализация до HS 7108 и учёт методологических причин.